In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os

# Adjust the path to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import weibull_min
from functions.functions_dp import *
import opendp.prelude as dp
import pickle
import seaborn as sns
import os
dp.enable_features("contrib")

# Read the data from the Excel file
df = pd.read_excel('../data/Gasverbräuche Bensheim_Etagenheizungen_grouped.xlsx')

# Using OpenDP library to remake the experiment on dp_initial notebook

In [ ]:
# Set the seed for reproducibility
np.random.seed(seed=0)

# Estimate the parameters for the Weibull distribution
k_estimate = 1
scale_estimate = np.median(df[2023])

print(f'Estimated shape parameter (c): {k_estimate} and scale parameter: {scale_estimate}')

# Parameters for the Weibull distribution
shape = k_estimate  # Shape parameter
scale = scale_estimate  # Scale parameter
n = 100 # Number of rows in the dataset

# Generate gas consumption data using the Weibull distribution
gas_consumption = weibull_min.rvs(shape, scale=scale, size=n)

# # Increase the values of gas consumption on the middle of the model
# for i in range (int(n/2)-3, int(n/2)+3):
#     gas_consumption[i] = gas_consumption[i] + (gas_consumption.max())

plot_index(gas_consumption)

In [ ]:
# Generate distance data using a normal distribution
distance_mean = 30
distance_std = 5
n = 100  # Sample size
distances = np.random.normal(loc=distance_mean, scale=distance_std, size=n)

# Define the font sizes
title_fontsize = 14
label_fontsize = 14
tick_fontsize = 12

plt.figure(figsize=(8, 4))
plt.hist(distances, bins=30, edgecolor='k', alpha=0.7)
plt.title('Distance Distribution', fontsize=title_fontsize)
plt.xlabel('Distance (m)', fontsize=label_fontsize)
plt.ylabel('Frequency', fontsize=label_fontsize)

# Set tick sizes
plt.tick_params(axis='x', labelsize=tick_fontsize)
plt.tick_params(axis='y', labelsize=tick_fontsize)

plt.show()

In [ ]:
df_generated = pd.DataFrame({
    'gas_consumption': gas_consumption,
    'distance': distances
})
df_generated['cumulative_distances'] = df_generated['distance'] / 2 + df_generated['distance'].cumsum().shift(fill_value=0)

heat_density_limit = 5000

df_generated['heat_density'] = df_generated['gas_consumption'] / df_generated['distance']
df_high_density = df_generated[df_generated['heat_density'] > heat_density_limit]

In [ ]:
epsilon = 1
print(f'Sensitivity: {gas_consumption.max() - gas_consumption.min()}')
sensitivity = gas_consumption.max() - gas_consumption.min()

Q1 = np.percentile(gas_consumption, 25)
Q3 = np.percentile(gas_consumption, 75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f'Upper bound: {upper_bound}')
bounds = (0., gas_consumption.max())
def a():
    base_lap = dp.m.make_laplace(
        dp.atom_domain(T=float),
        dp.absolute_distance(T=float),
        scale=sensitivity/epsilon
    )
    return base_lap
base_lap = a()
absolute_distance = sensitivity
print("epsilon:", base_lap.map(d_in=absolute_distance))

input_domain = dp.vector_domain(dp.atom_domain(bounds=bounds))
bounded_sum = dp.t.make_sum(input_domain, dp.symmetric_distance())

max_contributions = 1
print(bounded_sum.map(d_in=max_contributions))
bounded_sum.map(d_in=max_contributions)

input_domain = dp.vector_domain(dp.atom_domain(T=float))
input_metric = dp.symmetric_distance()

# call the constructor to produce the transformation `clamp`
clamp = dp.t.make_clamp(input_domain, input_metric, bounds=bounds)

dp_sum = clamp >> bounded_sum >> base_lap

In [ ]:
dp_sum = clamp >> bounded_sum >> base_lap

# compute the DP sum of a dataset of bounded elements
print("DP sum:", max(0, dp_sum(gas_consumption[0:40])))

# evaluate the privacy loss of the dp_sum, when an individual can contribute at most 2 records
print("epsilon:", dp_sum.map(d_in=1))

In [ ]:
def gen_laplacian_noise(epsilon, sensitivity, size=1):
    """
    Generate laplacian noise based on differential privacy params
    """

    # Calculate the scale parameter
    b = sensitivity / epsilon

    # Generate Laplacian noise
    noise = np.random.laplace(loc=0, scale=b, size=size)
    return noise
gen_laplacian_noise(1,1, 5)

In [ ]:
list_dfs = []
list_dfs_noisy = []
for i in range(1, n+1):
    df, noisy_df = calculate_grouped_heat_density(df_generated, n_groups=i, chain=dp_sum)
    list_dfs.append(df)
    list_dfs_noisy.append(noisy_df)

In [ ]:
plot_index(list_dfs[99]['heat_density'], list_dfs_noisy[99]['heat_density'], 'heat_density')

In [ ]:
plot_distances(list_dfs[10][0:10], 'heat_density')

In [ ]:
heat_density_limit = 2000
df_high_density_list = []
noisy_df_high_density_list = []
group_sizes = [1, 2, 5, 10, 20, 25, 50, 100]
plot_choice = 'heat_density' # 'heat_density' or 'gas_consumption'
for i in group_sizes:
    i=i-1
    df_high_density = list_dfs[i][list_dfs[i]['heat_density'] > heat_density_limit]
    df_high_density_list.append(df_high_density)
    noisy_df_high_density = list_dfs_noisy[i][list_dfs_noisy[i]['heat_density'] > heat_density_limit]
    noisy_df_high_density_list.append(noisy_df_high_density)
    accuracy = calculate_confusion_metrics(list_dfs[i], list_dfs_noisy[i], heat_density_limit)[0]
    plot_limit(list_dfs[i], list_dfs_noisy[i], df_high_density, noisy_df_high_density, i, f'../images/heat_density/group_size_{100/(i+1)}.png', plot_choice)

In [ ]:
# List to store the values for LaTeX table
table_data = []

# Loop to compute values and populate table_data
heat_density_limit = 2000
group_sizes = [1, 2, 5, 10, 20, 25, 50, 100]
for i in group_sizes:
    index = i - 1
    df_high_density = list_dfs[index][list_dfs[index]['heat_density'] > heat_density_limit]
    noisy_df_high_density = list_dfs_noisy[index][list_dfs_noisy[index]['heat_density'] > heat_density_limit]
    
    # Calculate accuracy
    accuracy = calculate_confusion_metrics(list_dfs[index], list_dfs_noisy[index], heat_density_limit)[0]
    
    # Calculate RMSE/Mean
    rmse_mean = (np.sqrt(mean_squared_error(list_dfs[index]['gas_consumption'], 
                                            list_dfs_noisy[index]['gas_consumption'])) 
                 / list_dfs[index]['gas_consumption'].mean() * 100)
    
    # Append values for LaTeX table
    table_data.append({
        "Group Size": f"{100 / i:.2f}",
        "Accuracy": f"{accuracy:.2f}",
        "RMSE/Mean (%)": f"{rmse_mean:.2f}"
    })

# Generate LaTeX table from table_data
latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|c|c|c|}\n\\hline\n"
latex_table += "Group Size & Accuracy & RMSE/Mean (\\%) \\\\\n\\hline\n"

for row in table_data:
    latex_table += f"{row['Group Size']} & {row['Accuracy']} & {row['RMSE/Mean (%)']} \\\\\n"

latex_table += "\\hline\n\\end{tabular}\n\\caption{Comparison of Group Sizes, Accuracy, and RMSE/Mean}\n\\label{tab:results}\n\\end{table}"

print(latex_table)

In [ ]:
heat_density_limit = 2000
df_high_density_list = []
noisy_df_high_density_list = []
group_sizes = [1, 2, 5, 10, 20, 25, 50, 100]
plot_choice = 'heat_density' # 'heat_density' or 'gas_consumption'
for i in group_sizes:
    i = i-1
    df_high_density = list_dfs[i][list_dfs[i]['heat_density'] > heat_density_limit]
    df_high_density_list.append(df_high_density)
    noisy_df_high_density = list_dfs_noisy[i][list_dfs_noisy[i]['heat_density'] > heat_density_limit]
    noisy_df_high_density_list.append(noisy_df_high_density)
    plot_limit_decision(list_dfs[i], list_dfs_noisy[i], df_high_density, noisy_df_high_density, size=i, file_name=f'images/heat_density/n_groups_{100/(i+1)}.png')

# Takes about 15 min to run with 25 GS, 500 epsilons and 10 repetitions

In [ ]:
group_sizes = [1, 2, 5, 10, 20, 25, 50, 100]
epsilon_values = np.logspace(-3, 3, num=500)
# Function to evaluate and save results
def evaluate_and_save(df_generated, group_sizes, epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, iterations, save_interval=2, filename='../data/all_results.pickle'):
    # Load existing results if the file exists
    if os.path.exists(filename):
        with open(filename, 'rb') as handle:
            all_results = pickle.load(handle)
        # Determine the last group size processed
        if not all_results.empty:
            last_group_size_done = all_results['n_groups'].max()
            start_index = list(group_sizes).index(last_group_size_done) + 1
        else:
            start_index = 0
    else:
        all_results = pd.DataFrame()
        start_index = 0
    
    # Continue evaluation from the next group size
    for i, group_size in enumerate(group_sizes[start_index:], start=start_index):
        # Evaluate for the current group size
        results = evaluate_over_epsilon(df_generated, [group_size], epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, iterations, calculate_grouped_heat_density)
        
        # Append the results to the all_results DataFrame
        all_results = pd.concat([all_results, pd.DataFrame(results)], ignore_index=True)
        
        # Save intermediate results every save_interval iterations
        if (i + 1) % save_interval == 0 or (i + 1) == len(group_sizes):
            with open(filename, 'wb') as handle:
                pickle.dump(all_results, handle, protocol=pickle.HIGHEST_PROTOCOL)
    return all_results

# Evaluate and save results
all_results = evaluate_and_save(df_generated, group_sizes, epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, 100)

plot_epsilon(all_results)
plot_rmse_mape(all_results)

In [ ]:
with open('../data/all_results.pickle', 'rb') as handle:
    all_results = pickle.load(handle)

In [ ]:
import matplotlib.ticker as ticker

plt.figure(figsize=(14, 6))
for n_groups in [5]:
    results = all_results[all_results['n_groups'] == n_groups]
    epsilon_values = np.array(results['epsilon'])
    rmse_mean = np.array(results['rmse_mean'])
    rmse_std = np.array(results['rmse_std'])
    rmse_perc_mean = np.array(results['rmse_perc_mean'])
    rmse_perc_std = np.array(results['rmse_perc_std'])
    mape_mean = np.array(results['mape_mean'])
    mape_std = np.array(results['mape_std'])
    accuracy_mean = np.array(results['accuracy_mean'])
    accuracy_std = np.array(results['accuracy_std'])

    # Plot RMSE
    plt.subplot(1, 2, 2)
    plt.xlim(10**-3, 1000)
    plt.ylim(0, 100)
    plt.fill_between(epsilon_values, np.clip(rmse_perc_mean - rmse_perc_std, 0, max(rmse_perc_mean - rmse_perc_std)), rmse_perc_mean + rmse_perc_std, color='r', alpha=0.2)
    plt.plot(epsilon_values, rmse_perc_mean, label=f'Group size = {int(100/n_groups)}', color='r')
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('RMSE / Mean (%)', fontsize=15)
    plt.title(f'RMSE / Mean per Epsilon', fontsize=16)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))
    plt.gca().yaxis.set_major_formatter(formatter)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.fill_between(epsilon_values, np.clip(accuracy_mean - accuracy_std, 0, 100), np.clip(accuracy_mean + accuracy_std, 0, 100), color='b', alpha=0.2)
    plt.plot(epsilon_values, accuracy_mean, label=f'Group size = {int(100/n_groups)}')
    plt.ylim(0, 105)
    plt.xlim(10**-3, 1000)
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('Accuracy (%)', fontsize=15)
    plt.title(f'Accuracy per Epsilon', fontsize=16)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

plt.tight_layout()
plt.show()

In [ ]:
plt.show()

In [ ]:
import matplotlib.ticker as ticker

plt.figure(figsize=(14, 6))
for n_groups in [1, 5, 10, 20, 25, 50, 100]:
    results = all_results[all_results['n_groups'] == n_groups]
    epsilon_values = np.array(results['epsilon'])
    rmse_mean = np.array(results['rmse_mean'])
    rmse_std = np.array(results['rmse_std'])
    rmse_perc_mean = np.array(results['rmse_perc_mean'])
    rmse_perc_std = np.array(results['rmse_perc_std'])
    mape_mean = np.array(results['mape_mean'])
    mape_std = np.array(results['mape_std'])
    accuracy_mean = np.array(results['accuracy_mean'])
    accuracy_std = np.array(results['accuracy_std'])

    # Plot RMSE
    plt.subplot(1, 2, 2)
    plt.xlim(10**-2, 1000)
    plt.ylim(0, 100)
    plt.plot(epsilon_values, rmse_perc_mean, label=f'Group size = {int(100/n_groups)}')
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('RMSE / Mean (%)', fontsize=15)
    plt.title('RMSE / Mean per Epsilon', fontsize=16)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))
    plt.gca().yaxis.set_major_formatter(formatter)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epsilon_values, accuracy_mean, label=f'Group size = {int(100/n_groups)}')
    plt.ylim(0, 105)
    plt.xlim(10**-2, 100)
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('Accuracy (%)', fontsize=15)
    plt.title('Accuracy per Epsilon', fontsize=16)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

plt.tight_layout()
plt.show()


In [ ]:
last_values = []
for n_groups in sorted(all_results['n_groups'].unique()):
    results = all_results[all_results['n_groups'] == n_groups]
    last_accuracy = results['accuracy_mean'].iloc[-1]
    last_rmse_perc = results['rmse_perc_mean'].iloc[-1]
    last_values.append((n_groups, last_accuracy, last_rmse_perc))

# Creating the LaTeX table
latex_table = r"""
\begin{table}[h!]
    \centering
    \begin{tabular}{|c|c|c|}
        \hline
        Group Size & Final Accuracy (\%) & Final RMSE/Mean (\%) \\
        \hline
"""

# Add rows to the LaTeX table
for group_size, accuracy, rmse in last_values:
    latex_table += f"        {100/group_size} & {accuracy} & {rmse} \\\\\n"

# Finalize the LaTeX table
latex_table += r"""        \hline
    \end{tabular}
    \caption{Final Accuracy and RMSE/Mean values for the last $\epsilon$ of each group size}
    \label{tab:final_accuracy_rmse}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)